### We will construct a linear model that can predict a insurance Premium amount by using its other attributes.

### Data Descrpition

The dataset has 7 variables.

1. age: age of the policyholder. 
2. 
sex: gender of the policyholde. r
3. 
bmi: Body Mass Index of the policyhold. e
4. r
children: number of children of the policyhol. d
5. er
smoker: whether the policyholder is a smoker or.  
6. not
region: region where the policyholder belon. g
7. s to
charges: premium charged to the policy. holder


### Import Libraries

In [ ]:
import pandas as pd
import numpy as np

# for visualizing data
import matplotlib.pyplot as plt
import seaborn as sns

# For randomized data splitting
from sklearn.model_selection import train_test_split

# To build linear regression_model
import statsmodels.api as sm

# To check model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error

## Load and explore the data

In [ ]:
cData = pd.read_csv("insurance.csv")

In [ ]:
cData.shape

In [ ]:
cData.head()

In [ ]:
cData.info()

* Most of the columns in the dataset are numeric in nature ('int64 or Float64')
* All are in correct datatypes

In [ ]:
# let's check the statistical summary of the data
cData.describe()

## Bivariate Analysis

A bivariate analysis among the different variables can be done using scatter matrix plot. Seaborn libs create a dashboard reflecting useful information about the dimensions. The result can be stored as a .png file.

In [ ]:
cData_attr = cData.iloc[:, 0:7]
sns.pairplot(
    cData_attr, diag_kind="kde"
) 

* By oberserving relationship between charge and other attributes are not really linear.
* serveral assumptions of classical linear regression seem to be vilotaed.

In [ ]:
# drop_first=True will drop one of the three origin columns
cData = pd.get_dummies(cData, columns=["sex"], drop_first=True)
cData.head()

In [ ]:
cData = pd.get_dummies(cData, columns=["smoker"], drop_first=True)
cData.head()

In [ ]:
cData = pd.get_dummies(cData, columns=["region"], drop_first=True)
cData.head()

In [ ]:
cData[["sex_male","smoker_yes","region_northwest","region_southeast","region_southwest"]] = cData[["sex_male","smoker_yes","region_northwest","region_southeast","region_southwest"]].replace({True: 1, False: 0})
cData.head()

In [ ]:
cData['log_bmi'] = np.log(cData['bmi'])

cData

In [ ]:
plt.hist(cData['log_bmi'], bins=50, label='Log-Transformed Data')
plt.legend()
plt.show()

In [ ]:
cData.count()

In [ ]:
data_numeric = cData.select_dtypes('number')
plt.figure(figsize=(10,5))
sns.heatmap(data_numeric.corr(),annot=True,cmap='Spectral',vmin=-1,vmax=1)
plt.show()

In [ ]:
cData = cData.drop(["log_bmi"], axis=1)

### Split Data

In [ ]:
# independent variables
X = cData.drop(["charges"], axis=1)
# dependent variable
y = cData[["charges"]]

In [ ]:
# let's add the intercept to data
X = sm.add_constant(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1
)

In [ ]:
print(X_train.head())

In [ ]:
print(X_test.shape)

## Fit Linear Model

In [ ]:
olsmod = sm.OLS(y_train, X_train)
olsres = olsmod.fit()

In [ ]:
# let's print the regression summary
print(olsres.summary())

In [ ]:
# let's check the VIF of the predictors
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_series1 = pd.Series(
    [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])],
    index=X_train.columns,
)
print("VIF values: \n\n{}\n".format(vif_series1))

In [ ]:
X_train2 = X_train.drop(["sex_male"], axis=1)
olsmod_2 = sm.OLS(y_train, X_train2)
olsres_2 = olsmod_2.fit()
print(olsres_2.summary())

In [ ]:
X_train3 = X_train.drop(["region_northwest"], axis=1)
olsmod_3 = sm.OLS(y_train, X_train3)
olsres_3 = olsmod_3.fit()
print(olsres_3.summary())

In [ ]:
X_train4 = X_train.drop(["region_southeast"], axis=1)
olsmod_4 = sm.OLS(y_train, X_train4)
olsres_4 = olsmod_4.fit()
print(olsres_4.summary())

In [ ]:
X_train5 = X_train.drop(["region_southeast"], axis=1)
olsmod_5 = sm.OLS(y_train, X_train5)
olsres_5 = olsmod_5.fit()
print(olsres_5.summary())

In [ ]:
df_pred = pd.DataFrame()

df_pred["Actual Values"] = y_train.values.flatten()  # actual values
df_pred["Fitted Values"] = olsres_5.fittedvalues.values  # predicted values
df_pred["Residuals"] = olsres_5.resid.values  # residuals

df_pred.head()

In [ ]:
# let us plot the fitted values vs residuals
sns.set_style("whitegrid")
sns.residplot(
    data=df_pred, x="Fitted Values", y="Residuals", color="purple", lowess=True
)
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Fitted vs Residual plot")
plt.show()

In [ ]:
import statsmodels.stats.api as sms
from statsmodels.compat import lzip

In [ ]:
name = ["F statistic", "p-value"]
test = sms.het_goldfeldquandt(df_pred["Residuals"], X_train5)
lzip(name, test)

In [ ]:
olsres_5.params